# Bot Detection Challenge — воспроизводимое решение

## Цель

Нужно для каждой `cookie_id` оценить вероятность того, что трафик относится к известным автоматизированным сервисам сбора данных (`target=1`). Основная метрика — **Precision при Recall ≥ 70%**.

В этом ноутбуке показан полный путь от исходных событий до итогового `submission.csv`. Код специально разбит на понятные этапы, чтобы было видно, какие данные используются, как строятся признаки и почему выбран именно такой способ валидации.

**Важное правило задачи:** признаки можно строить только из событий внутри индивидуального окна наблюдения: `window_start_ts <= event_ts < window_end_ts`.

**Воспроизводимость:** используются фиксированные `random_seed` для CatBoost (`0`, `42`, `123`), а итоговый score — среднее трёх моделей. Внешние API, LLM и ручная разметка теста не используются.


## 1. Загрузка библиотек и данных

Ожидаемая структура каталога при запуске ноутбука:

```text
data/
    train.csv
    test.csv
    events.csv.gz
metric.py
бот_detection_solution.ipynb
```

`train.csv` содержит целевую переменную, `test.csv` — нет, а `events.csv.gz` содержит историю событий. Временные столбцы сразу преобразуем в `datetime`, чтобы корректно фильтровать события по окнам.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from metric import precision_at_recall

SEED_LIST = [0, 42, 123]
DATA_DIR = Path("data")
OUTPUT = Path("submission.csv")

train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"]
)
test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"]
)
events = pd.read_csv(DATA_DIR / "events.csv.gz", parse_dates=["event_ts"])

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("Доля target=1 в train:", round(train.target.mean(), 4))


## 2. Первичный контроль данных

Перед построением признаков проверяем:

- уникальна ли `cookie_id` в train/test;
- какие типы событий есть;
- сколько пропусков в событиях;
- диапазон дат;
- есть ли дубликаты строк событий.

Это не только EDA: такие проверки помогают обнаружить ошибки, которые могли бы повлиять на признаки.


In [ ]:
print("Уникальность cookie_id:")
print("train:", train.cookie_id.is_unique)
print("test :", test.cookie_id.is_unique)

print("\nТипы событий:")
print(events.event_name.value_counts().head(20))

print("\nПлатформы:")
print(events.platform.value_counts(dropna=False))

print("\nДоля пропусков:")
print(events.isna().mean().sort_values(ascending=False).head(15).round(3))

print("\nДубликаты строк событий:", events.duplicated().sum())
print("Период событий:", events.event_ts.min(), "—", events.event_ts.max())


## 3. Временное окно наблюдения

Для каждой cookie есть своё начало и конец окна. Поэтому сначала присоединяем эти границы к событиям, а затем оставляем только события, удовлетворяющие условию:

`window_start_ts <= event_ts < window_end_ts`

Это принципиально важно: использование событий после окончания окна было бы утечкой информации из будущего.


In [ ]:
def events_in_window(events_all, meta):
    """Вернуть только события, попавшие в индивидуальное окно cookie."""
    e = events_all.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner"
    )
    return e[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)].copy()

ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)

print("Событий внутри train-окон:", len(ev_train))
print("Событий внутри test-окон :", len(ev_test))


## 4. Baseline из quickstart

Сначала нужен простой ориентир. Используем два признака из исходного quickstart:

1. количество событий cookie;
2. количество уникальных объявлений.

Для валидации используем **временной**, а не случайный split. Тестовое окно находится позже train, поэтому такой split лучше имитирует реальную проверку. Официальная функция `precision_at_recall` из `metric.py` используется без изменений.

Baseline нужен для ответа на вопрос: действительно ли сложные признаки дают дополнительную информацию.


In [ ]:
def basic_features(ev, meta):
    g = ev.groupby("cookie_id")
    f = pd.DataFrame({
        "n_events": g.size(),
        "item_nunique": g.item_id.nunique(),
    })
    return meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left").fillna(0)

X_basic = basic_features(ev_train, train)
y = train.target.to_numpy(dtype=int)

# Последняя часть train используется как имитация будущего периода.
is_valid = train.window_start_ts.ge("2026-04-17").values
cols_basic = ["n_events", "item_nunique"]

baseline = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=0
)
baseline.fit(X_basic.loc[~is_valid, cols_basic], y[~is_valid])
p_basic = baseline.predict_proba(X_basic.loc[is_valid, cols_basic])[:, 1]

print("Baseline P@R>=0.70:", round(precision_at_recall(y[is_valid], p_basic), 5))
print("Доля ботов на validation (уровень константного baseline):", round(y[is_valid].mean(), 5))


## 5. Построение расширенных поведенческих признаков

Основная идея улучшения — описать не только **сколько** событий произошло, но и **как именно** cookie взаимодействовала с сервисом.

Используются следующие группы признаков:

- **Интенсивность:** число событий и доли типов событий.
- **Разнообразие:** уникальные объявления, категории, локации, продавцы, запросы и страницы.
- **Пропуски:** насколько часто различные поля заполнены.
- **Время:** интервалы между событиями, короткие интервалы, длительность активности и число сессий.
- **Последовательность:** повторение одинаковых событий и разнообразие переходов между типами событий.
- **Поиск:** длина запроса, количество слов, номер страницы.
- **Курсор:** статистики перемещения указателя.
- **User-Agent:** компактные бинарные признаки платформы, браузера, ОС и технических токенов. Сырая строка UA в модель не передаётся.
- **Возраст cookie:** возраст на начало окна.
- **Недельная сезонность:** день недели начала окна. День месяца намеренно не используется.

Все эти признаки рассчитываются только после фильтрации по окну.


In [ ]:
def build_features(meta: pd.DataFrame, events_all: pd.DataFrame) -> pd.DataFrame:
    """Построить cookie-level признаки только из событий внутри окна наблюдения."""
    m = meta[["cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts"]].copy()
    e = events_all.merge(m, on="cookie_id", how="inner")
    e = e[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)].copy()
    e = e.sort_values(["cookie_id", "event_ts"], kind="mergesort")

    e["event_name"] = e["event_name"].fillna("NA")
    e["platform"] = e["platform"].fillna("NA").astype(str).str.lower()
    e["ua"] = e["user_agent"].fillna("NA").astype(str)

    g = e.groupby("cookie_id", sort=False)
    f = pd.DataFrame(index=m["cookie_id"])
    f.index.name = "cookie_id"

    # 1. Объём и состав событий.
    f["n_events"] = g.size()
    ev_counts = pd.crosstab(e.cookie_id, e.event_name)
    ev_counts.columns = ["cnt_" + str(c) for c in ev_counts.columns]
    f = f.join(ev_counts)
    for c in ev_counts.columns:
        f["share_" + c[4:]] = f[c] / f["n_events"]

    # 2. Разнообразие и заполненность полей.
    for col in ["item_id", "item_category", "item_location", "seller_type",
                "search_query", "search_page", "platform", "user_agent"]:
        f["nunique_" + col] = g[col].nunique(dropna=True)

    for col in ["item_id", "item_category", "item_location", "seller_type",
                "search_query", "search_page", "pointer_x", "pointer_y", "user_agent"]:
        nonnull = g[col].apply(lambda s: s.notna().sum())
        f["nonnull_" + col] = nonnull
        f["missing_share_" + col] = 1 - nonnull / f["n_events"]

    # 3. Числовой контекст поиска и курсора.
    for col in ["search_page", "pointer_x", "pointer_y"]:
        for stat in ["mean", "std", "max", "min"]:
            f[f"{col}_{stat}"] = g[col].agg(stat)

    # 4. Повторение одного объявления.
    item_counts = e.groupby(["cookie_id", "item_id"]).size()
    f["item_repeat_max"] = item_counts.groupby(level=0).max()
    f["item_repeat_mean"] = item_counts.groupby(level=0).mean()
    item_total = g["item_id"].count()
    f["item_repeat_frac"] = (
        item_counts[item_counts > 1].groupby(level=0).sum() / item_total
    )

    # 5. Время суток и день недели.
    e["hour"] = e.event_ts.dt.hour + e.event_ts.dt.minute / 60
    e["dow"] = e.event_ts.dt.dayofweek
    for h1, h2, name in [
        (0, 6, "night_0_6"), (6, 9, "morning_6_9"),
        (9, 18, "day_9_18"), (18, 24, "evening_18_24")
    ]:
        x = e[(e.event_ts.dt.hour >= h1) & (e.event_ts.dt.hour < h2)].groupby("cookie_id").size()
        f["cnt_" + name] = x
        f["share_" + name] = x / f["n_events"]

    f["hour_mean"] = g["hour"].mean()
    f["hour_std"] = g["hour"].std()
    f["dow_nunique"] = g["dow"].nunique()

    # 6. Интервалы между последовательными событиями.
    e["gap_sec"] = e.groupby("cookie_id").event_ts.diff().dt.total_seconds()
    gap = e.dropna(subset=["gap_sec"]).groupby("cookie_id")["gap_sec"]
    for stat in ["min", "median", "mean", "std", "max"]:
        f["gap_" + stat] = gap.agg(stat)

    for sec in [0.5, 1, 2, 5, 10, 30, 60]:
        x = e.groupby("cookie_id")["gap_sec"].apply(lambda s, q=sec: (s < q).sum())
        f[f"gaps_lt_{sec:g}s"] = x
        f[f"share_gaps_lt_{sec:g}s"] = x / (f["n_events"] - 1).replace(0, np.nan)

    span = g.event_ts.agg(lambda s: (s.max() - s.min()).total_seconds())
    f["active_span_sec"] = span
    f["events_per_active_hour"] = f["n_events"] / (span / 3600 + 1 / 3600)
    f["active_minutes"] = e.event_ts.dt.floor("min").groupby(e.cookie_id).nunique()
    f["sessions"] = e["gap_sec"].fillna(1e9).gt(30 * 60).groupby(e.cookie_id).sum()

    # 7. Последовательности событий.
    e["prev_event"] = e.groupby("cookie_id").event_name.shift()
    same = e.event_name == e.prev_event
    f["same_event_consecutive"] = same.groupby(e.cookie_id).sum()
    f["share_same_event_consecutive"] = f["same_event_consecutive"] / f["n_events"]
    trans = e["prev_event"].fillna("NA").astype(str) + "->" + e["event_name"].fillna("NA").astype(str)
    f["transition_nunique"] = e.assign(_trans=trans).groupby("cookie_id")["_trans"].nunique()

    # 8. Форма поисковых запросов.
    q = e["search_query"].fillna("").astype(str)
    e["query_len"] = q.str.len()
    e["query_tokens"] = q.str.split().str.len()
    for col in ["query_len", "query_tokens"]:
        for stat in ["mean", "std", "max", "min"]:
            f[f"{col}_{stat}"] = g[col].agg(stat)

    # 9. Движение курсора.
    e["dx"] = e.groupby("cookie_id").pointer_x.diff()
    e["dy"] = e.groupby("cookie_id").pointer_y.diff()
    e["move_dist"] = np.sqrt(e.dx**2 + e.dy**2)
    mv = e.groupby("cookie_id").move_dist
    for stat in ["mean", "std", "max", "median"]:
        f["pointer_move_" + stat] = mv.agg(stat)
    f["pointer_moves_nonzero"] = g["move_dist"].apply(lambda s: (s > 0).sum())

    # 10. Компактный разбор User-Agent.
    ua = e["ua"].str.lower()
    e["ua_mobile"] = ua.str.contains("android|iphone|ipad|mobile", regex=True).astype(int)
    e["ua_avito"] = ua.str.contains("avito").astype(int)
    e["ua_bot_tokens"] = ua.str.contains(
        "bot|crawler|spider|headless|selenium|playwright|puppeteer|httpclient|python-requests|curl|wget",
        regex=True
    ).astype(int)
    e["ua_chrome"] = ua.str.contains("chrome|crios").astype(int)
    e["ua_firefox"] = ua.str.contains("firefox").astype(int)
    e["ua_safari"] = ua.str.contains("safari").astype(int)
    e["ua_edge"] = ua.str.contains("edg/").astype(int)
    e["ua_yabrowser"] = ua.str.contains("yabrowser").astype(int)
    e["ua_linux"] = ua.str.contains("linux").astype(int)
    e["ua_windows"] = ua.str.contains("windows").astype(int)
    e["ua_mac"] = ua.str.contains("macintosh|mac os").astype(int)
    e["ua_android"] = ua.str.contains("android").astype(int)
    e["ua_ios"] = ua.str.contains("iphone|ipad").astype(int)
    for col in [
        "ua_mobile", "ua_avito", "ua_bot_tokens", "ua_chrome", "ua_firefox",
        "ua_safari", "ua_edge", "ua_yabrowser", "ua_linux", "ua_windows",
        "ua_mac", "ua_android", "ua_ios"
    ]:
        f[col + "_share"] = g[col].mean()

    # 11. Возраст cookie на начало окна.
    age = (
        m.set_index("cookie_id").window_start_ts -
        m.set_index("cookie_id").cookie_created_at
    ).dt.total_seconds() / 86400
    f["cookie_age_days"] = age

    # 12. День недели начала окна. День месяца намеренно не используем.
    f = f.reset_index().merge(
        m[["cookie_id", "window_start_ts"]], on="cookie_id", how="right"
    )
    f["window_dow"] = f["window_start_ts"].dt.dayofweek
    f = f.drop(columns=["window_start_ts"])

    numeric = [c for c in f.columns if c != "cookie_id"]
    f[numeric] = f[numeric].replace([np.inf, -np.inf], np.nan).fillna(0)
    return f


def add_mode_categories(X: pd.DataFrame, meta: pd.DataFrame, events_all: pd.DataFrame) -> pd.DataFrame:
    """Добавить наиболее частые категориальные значения внутри окна."""
    m = meta[["cookie_id", "window_start_ts", "window_end_ts"]]
    e = events_all.merge(m, on="cookie_id", how="inner")
    e = e[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)].copy()

    def mode_safe(s):
        s = s.dropna().astype(str)
        return "__NA__" if len(s) == 0 else s.mode().iloc[0]

    out = X.copy()
    gg = e.groupby("cookie_id")
    for col in ["platform", "user_agent", "item_category", "item_location",
                "seller_type", "search_query", "event_name"]:
        out["mode_" + col] = gg[col].agg(mode_safe)
    return out


## 6. Собираем train/test-признаки

Строим признаки отдельно для train и test, но сама функция одинаковая. Затем добавляем `mode_*` — наиболее частое значение некоторых категориальных полей в окне.

CatBoost умеет работать с категориальными признаками напрямую, поэтому эти значения не кодируются вручную через one-hot encoding.


In [ ]:
X_train = add_mode_categories(build_features(train, events), train, events)
X_test = add_mode_categories(build_features(test, events), test, events)

# Гарантируем одинаковый набор столбцов и порядок признаков.
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

feature_cols = [c for c in X_train.columns if c != "cookie_id"]
cat_cols = [c for c in feature_cols if c.startswith("mode_")]

for c in cat_cols:
    X_train[c] = X_train[c].fillna("__NA__").astype(str)
    X_test[c] = X_test[c].fillna("__NA__").astype(str)

print("Количество признаков:", len(feature_cols))
print("Из них категориальных:", len(cat_cols))
print("Размер X_train:", X_train.shape)
print("Размер X_test :", X_test.shape)


## 7. Временная валидация улучшенного решения

Случайный train/validation split здесь не использую: тест начинается позже обучающей выборки, поэтому случайное перемешивание может смешать близкие по времени наблюдения.

Для диагностики беру более позднюю часть train как validation. Модель обучается только на более ранних cookie, затем получает scores для более поздних. Важно, что сам `build_features` уже гарантирует отсутствие событий после конца каждого окна.

На validation оцениваем официальной метрикой `precision_at_recall`.


In [ ]:
# Один фиксированный seed используем для понятной и относительно быстрой диагностики.
# Финальная модель ниже всё равно будет ансамблем из трёх seed.
validation_seed = 42

model_val = CatBoostClassifier(
    iterations=800,
    depth=7,
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function="Logloss",
    random_seed=validation_seed,
    verbose=False,
    thread_count=-1,
    cat_features=cat_cols,
)

is_valid = train.window_start_ts.ge("2026-04-17").values

model_val.fit(
    X_train.loc[~is_valid, feature_cols],
    y[~is_valid]
)
p_val = model_val.predict_proba(X_train.loc[is_valid, feature_cols])[:, 1]

print("Improved model P@R>=0.70:", round(precision_at_recall(y[is_valid], p_val), 5))
print("Validation cookies:", int(is_valid.sum()))
print("Validation bots:", int(y[is_valid].sum()))


## 8. Почему итоговая модель — ансамбль из трёх CatBoost

После проверки признаков модель для финального submission обучается уже на **всём train**. Используются три запуска CatBoost с одинаковыми параметрами, но разными фиксированными seed: `0`, `42`, `123`.

Итоговый score cookie — среднее трёх вероятностей. Это не меняет формат задачи: проверяющей системе передаётся одно непрерывное значение от 0 до 1.

Порог отдельно не выбираем — согласно условию, на платформе он перебирается автоматически для расчёта `P@Recall≥0.70`.


In [ ]:
y = train.target.to_numpy(dtype=int)
predictions = []

for seed in SEED_LIST:
    print(f"Обучение CatBoost, seed={seed}...")
    model = CatBoostClassifier(
        iterations=800,
        depth=7,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="Logloss",
        random_seed=seed,
        verbose=False,
        thread_count=-1,
        cat_features=cat_cols,
    )
    model.fit(X_train[feature_cols], y)
    predictions.append(model.predict_proba(X_test[feature_cols])[:, 1])

score = np.mean(predictions, axis=0)

submission = pd.DataFrame({
    "cookie_id": test.cookie_id,
    "score": np.clip(score, 0, 1),
})

print(submission.score.describe())


## 9. Финальные проверки и сохранение `submission.csv`

Перед сохранением проверяем все требования формата:

- ровно две колонки;
- столько же строк, сколько в `test.csv`;
- каждая `cookie_id` встречается ровно один раз;
- порядок `cookie_id` совпадает с `test.csv`;
- score находится в диапазоне `[0, 1]`;
- пропусков нет.

После этих проверок файл записывается в корень проекта как `submission.csv`.


In [ ]:
assert list(submission.columns) == ["cookie_id", "score"]
assert len(submission) == len(test)
assert submission.cookie_id.is_unique
assert submission.cookie_id.equals(test.cookie_id)
assert submission.score.notna().all()
assert submission.score.between(0, 1).all()

submission.to_csv(OUTPUT, index=False)

print(f"Готово: {OUTPUT.resolve()}")
print("Строк:", len(submission))
print("Уникальных cookie:", submission.cookie_id.nunique())
print("Score в [0,1]:", submission.score.between(0, 1).all())
submission.head()


## 10. Итог и результат отправки

Полученный файл `submission.csv` соответствует формату задания. В отправленном на платформу решении фактический результат составил:

**Precision@Recall≥0.70 = 0.81407**

Для сравнения, константный ответ имел **0.09391**.

Эта цифра относится к фактической проверке отправленного `submission.csv`, а не к локальной временной валидации. Локальная валидация используется для проверки того, что признаки и схема обучения работают на будущем временном срезе без использования тестовой разметки.

### Что не использовалось

- ручная разметка тестовых cookie;
- хардкод ответов по конкретным `cookie_id`;
- внешние API;
- большие языковые модели;
- события за пределами окна наблюдения.

### Основная идея решения

Сильная сторона подхода — переход от двух простых агрегатов baseline к описанию **поведения cookie во времени**: интенсивности, последовательности событий, интервалов между действиями, разнообразия просмотренных объектов, поиска, движения курсора и User-Agent. Это позволяет модели отличать характер потока событий, а не только его объём.
